In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOKS 66/67'S REAL POLICY AND
#            RESULTS
# =============================================================================
import os
import sys
import gc
import json
import time
import importlib.util
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebooks 66/67's Real Policy and Results")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB66_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_66_summary.json"
NB67_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_67_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB66_SUMMARY_PATH, "run 66_profitability_modeling_business_understanding.ipynb first (Problem 13)"),
    (NB67_SUMMARY_PATH, "run 67_profitability_modeling_modeling.ipynb first (Problem 13)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB66_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB66_SUMMARY = json.load(f)
with open(NB67_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB67_SUMMARY = json.load(f)

POLICY_PATH = Path(NB66_SUMMARY["policy_path"])
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    PROFITABILITY_MODELING_POLICY = json.load(f)

MODELING_RESULTS_PATH = Path(NB67_SUMMARY["modeling_results_path"])
with open(MODELING_RESULTS_PATH, "r", encoding="utf-8") as f:
    MODELING_RESULTS = json.load(f)

PROFILE_PATH = Path(NB67_SUMMARY["profile_path"])
if not PROFILE_PATH.exists():
    raise FileNotFoundError(f"{PROFILE_PATH} not found.\nFix: re-run Notebook 67.")

REAL_SPEND_COLUMNS = PROFITABILITY_MODELING_POLICY["real_spend_columns"]
REVENUE_ASSUMPTIONS = PROFITABILITY_MODELING_POLICY["revenue_assumptions"]
AVG_MONTHLY_REVENUE_PER_ACCOUNT_USD = REVENUE_ASSUMPTIONS["avg_monthly_revenue_per_account_usd"]["value"]
REVENUE_MULTIPLIER_FLOOR = REVENUE_ASSUMPTIONS["revenue_multiplier_floor"]["value"]
REVENUE_MULTIPLIER_CEILING = REVENUE_ASSUMPTIONS["revenue_multiplier_ceiling"]["value"]
PROFITABILITY_TIER_NAMES = PROFITABILITY_MODELING_POLICY["profitability_tier_names"]
PROFITABILITY_TIER_CUT_PERCENTILES = PROFITABILITY_MODELING_POLICY["profitability_tier_cut_percentiles"]
KPI_TARGETS = PROFITABILITY_MODELING_POLICY["kpi_targets"]
P8_REUSE = PROFITABILITY_MODELING_POLICY["reused_from_problem_8"]
EAD_PER_ACCOUNT_USD = P8_REUSE["ead_per_account_usd"]
LGD_ASSUMPTION = P8_REUSE["lgd_assumption"]
P12_REUSE = PROFITABILITY_MODELING_POLICY["reused_from_problem_12"]
P12_PROFILE_PATH = Path(P12_REUSE["profile_path"])
if not P12_PROFILE_PATH.exists():
    raise FileNotFoundError(f"{P12_PROFILE_PATH} not found.\nFix: re-run Notebook 63 (Problem 12).")

REPORTED_CUT_LOW = NB67_SUMMARY["profitability_cut_low_usd"]
REPORTED_CUT_HIGH = NB67_SUMMARY["profitability_cut_high_usd"]
REPORTED_SPEARMAN_CORR = NB67_SUMMARY["risk_profitability_spearman_corr"]
REPORTED_RECOMMENDED_FOR_PRODUCTION = NB67_SUMMARY["recommended_for_production"]
RANDOM_SEED = NB67_SUMMARY["random_seed"]

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
DETECTED_TOTAL_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["total_ram_bytes_detected"]

P13_ROOT = PROJECT_ROOT / "Phase5_Customer_Business_Intelligence" / "Problem13_Risk_Adjusted_Profitability_Modeling"
if "profitability_modeling_validation_deployment" in PILLAR_DIRS:
    VALIDATION_DIR = PILLAR_DIRS["profitability_modeling_validation_deployment"]
else:
    VALIDATION_DIR = P13_ROOT / "validation_deployment"
    print(f"NOTE: 'profitability_modeling_validation_deployment' not in pillar_dirs -- using fallback: "
          f"{VALIDATION_DIR}")
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
API_SUBDIR = P13_ROOT / "src"
API_SUBDIR.mkdir(parents=True, exist_ok=True)
DOCS_SUBDIR = P13_ROOT / "docs"
DOCS_SUBDIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded Notebook 66 policy from : {POLICY_PATH}")
print(f"Loaded Notebook 67 results from: {MODELING_RESULTS_PATH}")
print(f"Loaded Notebook 67 profile from: {PROFILE_PATH}")
print(f"Reported PROFITABILITY_TIER cuts (Notebook 67): low=${REPORTED_CUT_LOW:,.2f}, "
      f"high=${REPORTED_CUT_HIGH:,.2f}")
print(f"Reported risk_adjustment_materiality Spearman (Notebook 67): {REPORTED_SPEARMAN_CORR:.4f} "
      f"(recommended_for_production: {REPORTED_RECOMMENDED_FOR_PRODUCTION})")
print(f"Validation artifacts will be written under: {VALIDATION_DIR}")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS (PHASE 5
#            TIGHTENED 92%/92% CAP + TWO-TIER RAM GUARD, REUSED VERBATIM)
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

_PHASE5_CPU_FRACTION_CAP = 0.92
_PHASE5_RAM_FRACTION_CAP = 0.92
_historical_thread_count = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
_historical_max_ram_bytes = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
WARP_THREAD_COUNT = min(_historical_thread_count, max(1, round(DETECTED_LOGICAL_CORES * _PHASE5_CPU_FRACTION_CAP)))
MAX_RAM_BYTES = min(_historical_max_ram_bytes, round(DETECTED_TOTAL_RAM_BYTES * _PHASE5_RAM_FRACTION_CAP))
os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    from scipy.stats import spearmanr
except ImportError:
    missing.append("scipy")
try:
    from fastapi.testclient import TestClient
except ImportError:
    missing.append("fastapi")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def _available_ram_gb() -> float:
    return psutil.virtual_memory().available / 1e9


_available_ram_gb_at_start = _available_ram_gb()
_comfortable_available_ram_gb = 0.50 * (MAX_RAM_BYTES / 1e9)
_min_required_available_ram_gb = 0.25 * (MAX_RAM_BYTES / 1e9)
if _available_ram_gb_at_start < _min_required_available_ram_gb:
    raise RuntimeError(
        f"Only {_available_ram_gb_at_start:.2f} GB of system RAM is available, below the "
        f"{_min_required_available_ram_gb:.2f} GB floor Section 4's reproduction needs. Close other "
        f"Jupyter kernels / applications, confirm with `psutil.virtual_memory().available / 1e9`, then "
        f"re-run this notebook from the top."
    )
if _available_ram_gb_at_start < _comfortable_available_ram_gb:
    print(f"⚠️  WARNING: only {_available_ram_gb_at_start:.2f} GB available "
          f"(comfortable margin {_comfortable_available_ram_gb:.2f} GB) -- proceeding.")
else:
    print(f"RAM pre-flight check passed: {_available_ram_gb_at_start:.2f} GB available.")

logger.info(f"Polars thread pool configured to {WARP_THREAD_COUNT}/{DETECTED_LOGICAL_CORES} threads "
            f"(Phase 5 tightened cap)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
print("\n✅ Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"train_split.csv     : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv      : {TEST_SPLIT_PATH}")
print("\n✅ Section 3 complete.")


# =============================================================================
# SECTION 4: INDEPENDENT REPRODUCTION OF NOTEBOOK 67'S PIPELINE
# =============================================================================
_section("SECTION 4: Independent Reproduction of Notebook 67's Pipeline")

# --- Rebuilds Notebook 67's entire real pipeline from scratch, in a fresh
#     kernel: reloads Problem 12's real unified profile, re-derives each
#     customer's real SPEND_PERCENTILE_RANK via a fresh streaming CSV pass,
#     re-computes REVENUE/PD_ADJUSTED_REVENUE/EXPECTED_LOSS/
#     PROFITABILITY_SCORE, re-fits the tertile cuts, and re-validates both
#     hard-gating KPIs -- the same independent-reproduction integrity check
#     Notebooks 48/52/56/64 already established for Problems 8/9/10/12. ---
P12_PROFILE_DF = pl.read_parquet(P12_PROFILE_PATH)
print(f"Reloaded Problem 12's real unified profile: {P12_PROFILE_DF.height:,} customers")

_spend_schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
for _c in REAL_SPEND_COLUMNS:
    _spend_schema_overrides[_c] = pl.Float32

print(f"Streaming the real raw CSV once (reproduction) for each customer's real latest-statement average "
      f"Spend signal. RSS: {_rss_gb():.2f} GB, available RAM: {_available_ram_gb():.2f} GB")
_t0 = time.time()
_REPRO_SPEND_DF = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH, schema_overrides=_spend_schema_overrides)
    .with_row_index("_csv_row_order")
    .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
    .sort(["customer_ID", "S_2", "_csv_row_order"])
    .group_by("customer_ID", maintain_order=False)
    .agg([pl.col(c).last().alias(c) for c in REAL_SPEND_COLUMNS])
    .with_columns(pl.mean_horizontal(REAL_SPEND_COLUMNS).alias("SPEND_RAW"))
    .select(["customer_ID", "SPEND_RAW"])
    .collect(engine="streaming")
)
print(f"Reproduced in {time.time() - _t0:.1f}s: {_REPRO_SPEND_DF.height:,} customers. "
      f"RSS: {_rss_gb():.2f} GB, available RAM: {_available_ram_gb():.2f} GB")

_repro_non_null = _REPRO_SPEND_DF.filter(pl.col("SPEND_RAW").is_not_null())
_repro_n_non_null = _repro_non_null.height
_repro_ranked = _repro_non_null.with_columns(
    ((pl.col("SPEND_RAW").rank(method="average") - 1.0) / max(_repro_n_non_null - 1, 1)).alias(
        "SPEND_PERCENTILE_RANK")
).select(["customer_ID", "SPEND_PERCENTILE_RANK"])
_REPRO_SPEND_DF = (
    _REPRO_SPEND_DF.join(_repro_ranked, on="customer_ID", how="left")
    .with_columns(pl.col("SPEND_PERCENTILE_RANK").fill_null(0.5))
    .select(["customer_ID", "SPEND_PERCENTILE_RANK"])
)
del _repro_non_null, _repro_ranked
gc.collect()

REPRO_DF = P12_PROFILE_DF.join(_REPRO_SPEND_DF, on="customer_ID", how="left").with_columns(
    pl.col("SPEND_PERCENTILE_RANK").fill_null(0.5)
).with_columns([
    (REVENUE_MULTIPLIER_FLOOR + (REVENUE_MULTIPLIER_CEILING - REVENUE_MULTIPLIER_FLOOR)
     * pl.col("SPEND_PERCENTILE_RANK")).alias("REVENUE_MULTIPLIER"),
]).with_columns([
    (AVG_MONTHLY_REVENUE_PER_ACCOUNT_USD * pl.col("REVENUE_MULTIPLIER")).alias("REVENUE_PER_ACCOUNT_USD"),
]).with_columns([
    (pl.col("REVENUE_PER_ACCOUNT_USD") * (1.0 - pl.col("UNIFIED_RISK_SCORE"))).alias("PD_ADJUSTED_REVENUE_USD"),
    (pl.col("UNIFIED_RISK_SCORE") * EAD_PER_ACCOUNT_USD * LGD_ASSUMPTION).alias("EXPECTED_LOSS_USD"),
]).with_columns([
    (pl.col("PD_ADJUSTED_REVENUE_USD") - pl.col("EXPECTED_LOSS_USD")).alias("PROFITABILITY_SCORE"),
])

TRAIN_IDS_DF = pl.read_csv(TRAIN_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
TEST_IDS_DF = pl.read_csv(TEST_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
REPRO_TRAIN_DF = REPRO_DF.join(TRAIN_IDS_DF, on="customer_ID", how="inner")
REPRO_HOLDOUT_DF = REPRO_DF.join(TEST_IDS_DF, on="customer_ID", how="inner")

_p_lo, _p_hi = PROFITABILITY_TIER_CUT_PERCENTILES[0] / 100.0, PROFITABILITY_TIER_CUT_PERCENTILES[1] / 100.0
REPRODUCED_CUT_LOW = float(REPRO_TRAIN_DF["PROFITABILITY_SCORE"].quantile(_p_lo))
REPRODUCED_CUT_HIGH = float(REPRO_TRAIN_DF["PROFITABILITY_SCORE"].quantile(_p_hi))

_repro_risk_arr = REPRO_HOLDOUT_DF["UNIFIED_RISK_SCORE"].to_numpy()
_repro_profit_arr = REPRO_HOLDOUT_DF["PROFITABILITY_SCORE"].to_numpy()
REPRODUCED_SPEARMAN_CORR, _ = spearmanr(_repro_risk_arr, _repro_profit_arr)
REPRODUCED_SPEARMAN_CORR = float(REPRODUCED_SPEARMAN_CORR)

_cut_low_diff = abs(REPRODUCED_CUT_LOW - REPORTED_CUT_LOW)
_cut_high_diff = abs(REPRODUCED_CUT_HIGH - REPORTED_CUT_HIGH)
_spearman_diff = abs(REPRODUCED_SPEARMAN_CORR - REPORTED_SPEARMAN_CORR)
print(f"\nReproduced cut low / high    : ${REPRODUCED_CUT_LOW:,.4f} / ${REPRODUCED_CUT_HIGH:,.4f}  "
      f"(Notebook 67 reported ${REPORTED_CUT_LOW:,.4f} / ${REPORTED_CUT_HIGH:,.4f})")
print(f"Reproduced Spearman correlation: {REPRODUCED_SPEARMAN_CORR:.6f}  "
      f"(Notebook 67 reported {REPORTED_SPEARMAN_CORR:.6f})")
REPRODUCTION_PASSED = bool(_cut_low_diff < 0.01 and _cut_high_diff < 0.01 and _spearman_diff < 1e-4)
print(f"Reproduction match (cut tolerance $0.01, correlation tolerance 1e-4): "
      f"{'PASS' if REPRODUCTION_PASSED else 'FAIL'}")
if not REPRODUCTION_PASSED:
    raise AssertionError(
        "Notebook 68's independent reproduction of Notebook 67's pipeline does NOT match within tolerance -- "
        "investigate before proceeding. Do not deploy this artifact until this is resolved."
    )
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: CROSS-CHECK THE PERSISTED PROFILE AGAINST A FRESH REPRODUCTION
#            (REAL SAMPLE OF HOLDOUT CUSTOMERS)
# =============================================================================
_section("SECTION 5: Cross-Check the Persisted Profile Against a Fresh Reproduction")


def _assign_tier(df: "pl.DataFrame", cut_low: float, cut_high: float) -> "pl.DataFrame":
    _tier_expr = (
        pl.when(pl.col("PROFITABILITY_SCORE") <= cut_low).then(pl.lit(PROFITABILITY_TIER_NAMES[0]))
        .when(pl.col("PROFITABILITY_SCORE") <= cut_high).then(pl.lit(PROFITABILITY_TIER_NAMES[1]))
        .otherwise(pl.lit(PROFITABILITY_TIER_NAMES[2]))
        .alias("PROFITABILITY_TIER_REPRO")
    )
    return df.with_columns(_tier_expr)


REPRO_HOLDOUT_DF = _assign_tier(REPRO_HOLDOUT_DF, REPRODUCED_CUT_LOW, REPRODUCED_CUT_HIGH)

PERSISTED_PROFILE_DF = pl.read_parquet(PROFILE_PATH)
_n_sample = min(500, REPRO_HOLDOUT_DF.height)
_rng_sample = np.random.default_rng(RANDOM_SEED)
_sample_idx = _rng_sample.choice(REPRO_HOLDOUT_DF.height, size=_n_sample, replace=False)
_sample_ids = REPRO_HOLDOUT_DF[_sample_idx.tolist()].select(
    ["customer_ID", "PROFITABILITY_SCORE", "PROFITABILITY_TIER_REPRO"]
).rename({"PROFITABILITY_SCORE": "PROFITABILITY_SCORE_repro"})

_compare = _sample_ids.join(
    PERSISTED_PROFILE_DF.select(["customer_ID", "PROFITABILITY_SCORE", "PROFITABILITY_TIER"]),
    on="customer_ID", how="inner",
)
_max_sample_diff = float((_compare["PROFITABILITY_SCORE_repro"] - _compare["PROFITABILITY_SCORE"]).abs().max()) \
    if _compare.height else float("nan")
_tier_mismatches = int((_compare["PROFITABILITY_TIER_REPRO"] != _compare["PROFITABILITY_TIER"]).sum())
PROFILE_VERIFIED = bool(
    _compare.height == _sample_ids.height and _max_sample_diff < 0.01 and _tier_mismatches == 0
)
print(f"Persisted profile cross-check ({_compare.height} sampled holdout customers): "
      f"max PROFITABILITY_SCORE diff ${_max_sample_diff:.6f}, tier mismatches {_tier_mismatches} -- "
      f"{'PASS' if PROFILE_VERIFIED else 'FAIL'}")
if not PROFILE_VERIFIED:
    raise AssertionError(
        "The persisted profitability-scored profile does NOT match a fresh reproduction for the sampled "
        "customers -- do not deploy this artifact until this is resolved."
    )
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: BOOTSTRAP CONFIDENCE INTERVAL -- RISK_ADJUSTMENT_MATERIALITY
#            SPEARMAN CORRELATION
# =============================================================================
_section("SECTION 6: Bootstrap Confidence Interval -- risk_adjustment_materiality Spearman Correlation")

# --- profitability_tier_monotonicity's per-tier default rate IS a genuinely
#     sampling-variable statistic too, but Notebook 64 already established
#     the platform's precedent of bootstrapping the single KPI most central
#     to this problem's own new claim -- here, that PD-adjustment measurably
#     re-ranks customers (the Spearman correlation), consistent with
#     Notebook 64's honest scoping of its own bootstrap section. ---
_rng = np.random.default_rng(RANDOM_SEED)
_n_boot = 200
_n_holdout = len(_repro_risk_arr)
_boot_corrs = np.empty(_n_boot, dtype=np.float64)
for _i in range(_n_boot):
    _idx = _rng.integers(0, _n_holdout, size=_n_holdout)
    _corr_b, _ = spearmanr(_repro_risk_arr[_idx], _repro_profit_arr[_idx])
    _boot_corrs[_i] = _corr_b if np.isfinite(_corr_b) else np.nan

_boot_corrs_valid = _boot_corrs[~np.isnan(_boot_corrs)]
SPEARMAN_CORR_CI = [
    float(np.percentile(_boot_corrs_valid, 2.5)), float(np.percentile(_boot_corrs_valid, 97.5))
] if len(_boot_corrs_valid) > 0 else [float("nan"), float("nan")]
print(f"Bootstrap 95% CI on the risk_adjustment_materiality Spearman correlation "
      f"({len(_boot_corrs_valid)} valid resamples of {_n_boot}): "
      f"[{SPEARMAN_CORR_CI[0]:.4f}, {SPEARMAN_CORR_CI[1]:.4f}]")

_spearman_threshold = KPI_TARGETS["risk_adjustment_materiality"]["spearman_threshold"]
RISK_ADJUSTMENT_MATERIALITY_CI_PASSED = bool(SPEARMAN_CORR_CI[1] <= _spearman_threshold)
print(f"\nrisk_adjustment_materiality CI upper bound <= {_spearman_threshold} (entire CI stays materially "
      f"negative): {RISK_ADJUSTMENT_MATERIALITY_CI_PASSED}")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: HONEST LIMITATION -- DEPLOYMENT SCOPE & ASSUMPTIONS
# =============================================================================
_section("SECTION 7: Honest Limitation -- Deployment Scope & Assumptions")

_repro_tier_default_rates = {}
for _tier in PROFITABILITY_TIER_NAMES:
    _tdf = REPRO_HOLDOUT_DF.filter(pl.col("PROFITABILITY_TIER_REPRO") == _tier)
    _repro_tier_default_rates[_tier] = float(_tdf["target"].mean()) if _tdf.height else float("nan")
_ordered_rates = [_repro_tier_default_rates[t] for t in PROFITABILITY_TIER_NAMES]
PROFITABILITY_TIER_MONOTONICITY_REPRO_PASSED = bool(_ordered_rates[0] >= _ordered_rates[1] >= _ordered_rates[2])

MEETS_KPI_WITH_CI = bool(RISK_ADJUSTMENT_MATERIALITY_CI_PASSED and PROFITABILITY_TIER_MONOTONICITY_REPRO_PASSED)
print(
    "DEPLOYMENT SCOPE (honest): this service serves Problem 13's real precomputed profitability score -- a "
    "lookup by customer_ID, not a live-compute-from-inputs endpoint (the same architecture choice Notebook 64 "
    "made for Problem 12, for the same reason: recomputing the full pipeline per request would mean "
    "re-implementing Problem 12's own four-signal composite a second time inside this service).\n\n"
    "The dollar figures this service returns (REVENUE_PER_ACCOUNT_USD, PD_ADJUSTED_REVENUE_USD, "
    "EXPECTED_LOSS_USD, PROFITABILITY_SCORE) are NOT audited real financial figures -- REVENUE_PER_ACCOUNT_USD "
    "in particular rests on Notebook 66's explicit ASSUMPTION average-monthly-revenue and multiplier range, "
    "scaled by a real, measured relative Spend rank. Every report this problem produces states this plainly; "
    "this service's own /policy-info endpoint surfaces the same ASSUMPTION values so no caller can mistake "
    "them for measured fact.\n\n"
    f"RECOMMENDED FOR PRODUCTION: {MEETS_KPI_WITH_CI} (risk_adjustment_materiality's 95% bootstrap CI "
    f"{'holds' if RISK_ADJUSTMENT_MATERIALITY_CI_PASSED else 'does NOT hold'} on the real HOLDOUT split; "
    f"profitability_tier_monotonicity {'passed' if PROFITABILITY_TIER_MONOTONICITY_REPRO_PASSED else 'did NOT pass'} "
    "on this notebook's own independent reproduction)."
)
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: PERSIST DEPLOYMENT POLICY ARTIFACT
# =============================================================================
_section("SECTION 8: Persist Deployment Policy Artifact")

PROFITABILITY_DEPLOYMENT_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "profitability_tier_names": PROFITABILITY_TIER_NAMES,
    "profitability_cut_low_usd": REPRODUCED_CUT_LOW,
    "profitability_cut_high_usd": REPRODUCED_CUT_HIGH,
    "revenue_assumptions": REVENUE_ASSUMPTIONS,
    "reported_spearman_correlation": REPORTED_SPEARMAN_CORR,
    "reproduced_spearman_correlation": REPRODUCED_SPEARMAN_CORR,
    "reproduction_passed": REPRODUCTION_PASSED,
    "profile_verified": PROFILE_VERIFIED,
    "spearman_correlation_ci_95": SPEARMAN_CORR_CI,
    "risk_adjustment_materiality_ci_passed": RISK_ADJUSTMENT_MATERIALITY_CI_PASSED,
    "profitability_tier_monotonicity_passed": PROFITABILITY_TIER_MONOTONICITY_REPRO_PASSED,
    "meets_kpi_with_ci": MEETS_KPI_WITH_CI,
    "recommended_for_production": bool(MEETS_KPI_WITH_CI and REPRODUCTION_PASSED and PROFILE_VERIFIED),
    "ead_per_account_usd": EAD_PER_ACCOUNT_USD, "lgd_assumption": LGD_ASSUMPTION,
    "profile_path": str(PROFILE_PATH),
    "random_seed": RANDOM_SEED,
}
deployment_policy_path = DOCS_SUBDIR / "profitability_deployment_policy.json"
with open(deployment_policy_path, "w", encoding="utf-8") as f:
    json.dump(PROFITABILITY_DEPLOYMENT_POLICY, f, indent=2)
print(f"Wrote: {deployment_policy_path}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: GENERATE profitability_scoring_lookup_service.py -- REAL,
#            RUNNABLE FASTAPI LOOKUP SERVICE WITH AUTH + EXPLAINABILITY
# =============================================================================
_section("SECTION 9: Generate profitability_scoring_lookup_service.py")

_policy_path_str = str(deployment_policy_path)
_profile_path_str = str(PROFILE_PATH)

PROFITABILITY_SERVICE_TEMPLATE = "\n".join([
    "# AMEX Enterprise Credit Risk Platform -- Risk-Adjusted Profitability Scoring Lookup API.",
    "# Auto-generated by 68_profitability_modeling_validation_deployment.ipynb.",
    "# Serves each customer's real precomputed PROFITABILITY_SCORE / PROFITABILITY_TIER, composed from",
    "# Problem 12's real UNIFIED_RISK_SCORE and an ASSUMPTION-labeled revenue estimate (Notebook 66).",
    "# Every endpoint except /health requires a valid X-API-Key header (see .env.example).",
    "# Run with:",
    "#     uvicorn profitability_scoring_lookup_service:app --host 0.0.0.0 --port 8013",
    "import json",
    "import logging",
    "import os",
    "import secrets",
    "from pathlib import Path",
    "from typing import List",
    "",
    "import polars as pl",
    "from fastapi import Depends, FastAPI, HTTPException, Security",
    "from fastapi.security import APIKeyHeader",
    "from pydantic import BaseModel",
    "",
    "_auth_logger = logging.getLogger(__name__ + \".auth\")",
    "_DEV_DEFAULT_API_KEY = \"dev-only-CHANGE-ME-before-deploying\"",
    "_api_key_header = APIKeyHeader(name=\"X-API-Key\", auto_error=False)",
    "",
    "",
    "def _configured_api_key() -> str:",
    "    key = os.environ.get(\"API_KEY\")",
    "    if not key:",
    "        _auth_logger.warning(",
    "            \"API_KEY is not set -- falling back to the published dev-only default. Set API_KEY \"",
    "            \"before deploying this service anywhere reachable by anyone but you.\"",
    "        )",
    "        return _DEV_DEFAULT_API_KEY",
    "    return key",
    "",
    "",
    "def require_api_key(presented: str = Security(_api_key_header)) -> str:",
    "    expected = _configured_api_key()",
    "    if not presented or not secrets.compare_digest(presented, expected):",
    "        raise HTTPException(status_code=401, detail=\"Missing or invalid X-API-Key header.\")",
    "    return presented",
    "",
    "",
    "POLICY_PATH = Path(os.environ.get(\"AMEX_P13_POLICY_PATH\", r\"__POLICY_PATH_TOKEN__\"))",
    "PROFILE_PATH = Path(os.environ.get(\"AMEX_P13_PROFILE_PATH\", r\"__PROFILE_PATH_TOKEN__\"))",
    "with open(POLICY_PATH, \"r\", encoding=\"utf-8\") as _f:",
    "    _POLICY = json.load(_f)",
    "",
    "PROFITABILITY_TIER_NAMES = _POLICY[\"profitability_tier_names\"]",
    "CUT_LOW = _POLICY[\"profitability_cut_low_usd\"]",
    "CUT_HIGH = _POLICY[\"profitability_cut_high_usd\"]",
    "REVENUE_ASSUMPTIONS = _POLICY[\"revenue_assumptions\"]",
    "RECOMMENDED_FOR_PRODUCTION = _POLICY[\"recommended_for_production\"]",
    "",
    "_PROFILE_DF = pl.read_parquet(PROFILE_PATH)",
    "_PROFILE_INDEX = {row[\"customer_ID\"]: row for row in _PROFILE_DF.iter_rows(named=True)}",
    "",
    "",
    "class ProfitabilityResponse(BaseModel):",
    "    customer_id: str",
    "    unified_risk_score: float",
    "    spend_percentile_rank: float",
    "    revenue_per_account_usd: float",
    "    pd_adjusted_revenue_usd: float",
    "    expected_loss_usd: float",
    "    profitability_score_usd: float",
    "    profitability_tier: str",
    "    rationale: str",
    "    reasoning: List[str] = []",
    "    recommended_for_production: bool = RECOMMENDED_FOR_PRODUCTION",
    "",
    "",
    "app = FastAPI(",
    "    title=\"AMEX Enterprise Credit Risk Platform -- Risk-Adjusted Profitability Scoring Lookup API\",",
    "    description=\"Serves each real customer's precomputed PD-adjusted profitability score. Dollar figures \"",
    "                \"rest partly on explicit ASSUMPTION revenue inputs -- see /policy-info. Every endpoint \"",
    "                \"except /health requires a valid X-API-Key header.\",",
    "    version=\"1.0.0\",",
    ")",
    "",
    "",
    "@app.get(\"/health\")",
    "def health():",
    "    return {\"status\": \"ok\", \"profiles_loaded\": len(_PROFILE_INDEX)}",
    "",
    "",
    "@app.get(\"/policy-info\", dependencies=[Depends(require_api_key)])",
    "def policy_info():",
    "    return {",
    "        \"profitability_tier_names\": PROFITABILITY_TIER_NAMES,",
    "        \"profitability_tier_cuts_usd\": [CUT_LOW, CUT_HIGH],",
    "        \"revenue_assumptions\": REVENUE_ASSUMPTIONS,",
    "        \"recommended_for_production\": RECOMMENDED_FOR_PRODUCTION,",
    "        \"profiles_loaded\": len(_PROFILE_INDEX),",
    "    }",
    "",
    "",
    "@app.get(\"/profitability/{customer_id}\", response_model=ProfitabilityResponse, "
    "dependencies=[Depends(require_api_key)])",
    "def get_profitability(customer_id: str):",
    "    row = _PROFILE_INDEX.get(customer_id)",
    "    if row is None:",
    "        raise HTTPException(status_code=404, detail=f\"No profitability profile found for "
    "customer_id={customer_id!r}.\")",
    "    reasoning = [",
    "        f\"unified_risk_score={row['UNIFIED_RISK_SCORE']:.4f} -> revenue is PD-adjusted by \"",
    "        f\"(1 - {row['UNIFIED_RISK_SCORE']:.4f}) = {1.0 - row['UNIFIED_RISK_SCORE']:.4f}\",",
    "        f\"spend_percentile_rank={row['SPEND_PERCENTILE_RANK']:.4f} (real, measured) -> \"",
    "        f\"revenue_multiplier={row['REVENUE_MULTIPLIER']:.4f} -> \"",
    "        f\"revenue_per_account_usd=${row['REVENUE_PER_ACCOUNT_USD']:.2f} (ASSUMPTION-scaled)\",",
    "        f\"expected_loss_usd=${row['EXPECTED_LOSS_USD']:.2f} \"",
    "        f\"(unified_risk_score x EAD x LGD, EAD/LGD inherited from Notebook 08)\",",
    "        f\"profitability_score_usd=${row['PROFITABILITY_SCORE']:.2f} -> {row['PROFITABILITY_TIER']} \"",
    "        f\"(cuts: <= ${CUT_LOW:.2f} {PROFITABILITY_TIER_NAMES[0]}, \"",
    "        f\"<= ${CUT_HIGH:.2f} {PROFITABILITY_TIER_NAMES[1]}, else {PROFITABILITY_TIER_NAMES[2]})\",",
    "    ]",
    "    return ProfitabilityResponse(",
    "        customer_id=row[\"customer_ID\"], unified_risk_score=row[\"UNIFIED_RISK_SCORE\"],",
    "        spend_percentile_rank=row[\"SPEND_PERCENTILE_RANK\"],",
    "        revenue_per_account_usd=row[\"REVENUE_PER_ACCOUNT_USD\"],",
    "        pd_adjusted_revenue_usd=row[\"PD_ADJUSTED_REVENUE_USD\"], expected_loss_usd=row[\"EXPECTED_LOSS_USD\"],",
    "        profitability_score_usd=row[\"PROFITABILITY_SCORE\"], profitability_tier=row[\"PROFITABILITY_TIER\"],",
    "        rationale=f\"PD-adjusted revenue (real relative spend rank, ASSUMPTION dollar scale) minus \"",
    "                  f\"expected loss (real unified risk score x Notebook 08's real EAD/LGD).\",",
    "        reasoning=reasoning,",
    "    )",
    "",
])
PROFITABILITY_SERVICE_SOURCE = (
    PROFITABILITY_SERVICE_TEMPLATE
    .replace("__POLICY_PATH_TOKEN__", _policy_path_str)
    .replace("__PROFILE_PATH_TOKEN__", _profile_path_str)
)

service_py_path = API_SUBDIR / "profitability_scoring_lookup_service.py"
with open(service_py_path, "w", encoding="utf-8") as f:
    f.write(PROFITABILITY_SERVICE_SOURCE)
compile(PROFITABILITY_SERVICE_SOURCE, str(service_py_path), "exec")
print(f"Generated {len(PROFITABILITY_SERVICE_SOURCE.splitlines())} lines, syntax-checked OK.")
print(f"Saved -> {service_py_path}")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: LIVE SELF-TEST -- IMPORT THE GENERATED SERVICE & DRIVE IT WITH
#             REAL HOLDOUT CUSTOMERS' ACTUAL PROFILES
# =============================================================================
_section("SECTION 10: Live Self-Test -- Import the Generated Service & Drive It")

os.environ["AMEX_P13_POLICY_PATH"] = str(deployment_policy_path)
os.environ["AMEX_P13_PROFILE_PATH"] = str(PROFILE_PATH)
_TEST_API_KEY = "pytest-only-test-key"
os.environ["API_KEY"] = _TEST_API_KEY
_spec = importlib.util.spec_from_file_location("amex_profitability_service", str(service_py_path))
_service_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_service_module)
client = TestClient(_service_module.app)
_auth_headers = {"X-API-Key": _TEST_API_KEY}

_health_resp = client.get("/health")
assert _health_resp.status_code == 200
print(f"GET /health              (no key)   -> {_health_resp.status_code}  {_health_resp.json()}")

_unauth_resp = client.get("/policy-info")
assert _unauth_resp.status_code == 401
print(f"GET /policy-info         (no key, should reject) -> {_unauth_resp.status_code}")

_info_resp = client.get("/policy-info", headers=_auth_headers)
assert _info_resp.status_code == 200
print(f"GET /policy-info         (with key) -> {_info_resp.status_code}  "
      f"recommended_for_production={_info_resp.json()['recommended_for_production']}")

# Real end-to-end check against 3 real holdout customers, one per real tier.
API_SELF_TEST_ROWS_PASSED = []
_sample_customers = []
for _tier in PROFITABILITY_TIER_NAMES:
    _tdf = REPRO_HOLDOUT_DF.filter(pl.col("PROFITABILITY_TIER_REPRO") == _tier)
    if _tdf.height:
        _sample_customers.append(_tdf.row(_tdf.height // 2, named=True))

for _row in _sample_customers:
    _resp = client.get(f"/profitability/{_row['customer_ID']}", headers=_auth_headers)
    assert _resp.status_code == 200, f"/profitability returned {_resp.status_code}: {_resp.text}"
    _result = _resp.json()
    _row_passed = (
        _result["profitability_tier"] == _row["PROFITABILITY_TIER_REPRO"]
        and abs(_result["profitability_score_usd"] - _row["PROFITABILITY_SCORE"]) < 0.5
    )
    API_SELF_TEST_ROWS_PASSED.append(_row_passed)
    print(f"GET /profitability/{_row['customer_ID']}: API tier={_result['profitability_tier']} "
          f"(expected {_row['PROFITABILITY_TIER_REPRO']}), score=${_result['profitability_score_usd']:.2f} "
          f"(expected ${_row['PROFITABILITY_SCORE']:.2f}) -- {'PASS' if _row_passed else 'FAIL'}")

_unauth_profile_resp = client.get(f"/profitability/{_sample_customers[0]['customer_ID']}")
assert _unauth_profile_resp.status_code == 401, "/profitability without a key should be rejected"
print(f"GET /profitability/...    (no key, should reject) -> {_unauth_profile_resp.status_code}")

_missing_resp = client.get("/profitability/CUSTOMER_ID_THAT_DOES_NOT_EXIST", headers=_auth_headers)
assert _missing_resp.status_code == 404, "/profitability for an unknown customer should 404"
print(f"GET /profitability/<unknown> (should 404) -> {_missing_resp.status_code}")

API_SELF_TEST_PASSED = bool(all(API_SELF_TEST_ROWS_PASSED)) and len(API_SELF_TEST_ROWS_PASSED) == len(
    PROFITABILITY_TIER_NAMES)
if not API_SELF_TEST_PASSED:
    raise RuntimeError("Notebook 68's API self-test FAILED -- see checks above. Not safe to proceed.")
print("\n✅ Section 10 complete -- auth rejects unkeyed calls, all sampled real holdout customers' profiles "
      "match direct computation, an unknown customer_id correctly 404s.")


# =============================================================================
# SECTION 11: GENERATE .env.example & requirements-api.txt
# =============================================================================
_section("SECTION 11: Generate .env.example & requirements-api.txt")

_env_example = "\n".join([
    "# Copy to .env and fill in real values before deploying.",
    "API_KEY=dev-only-CHANGE-ME-before-deploying",
    f"AMEX_P13_POLICY_PATH={deployment_policy_path}",
    f"AMEX_P13_PROFILE_PATH={PROFILE_PATH}",
    "",
])
(API_SUBDIR / ".env.example").write_text(_env_example, encoding="utf-8")
_requirements_api = "\n".join(["fastapi>=0.110", "uvicorn>=0.29", "pydantic>=2.0", "polars>=0.20", ""])
(API_SUBDIR / "requirements-api.txt").write_text(_requirements_api, encoding="utf-8")
print(f"Wrote: {API_SUBDIR / '.env.example'}")
print(f"Wrote: {API_SUBDIR / 'requirements-api.txt'}")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 12: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Deployment policy file was written", deployment_policy_path.exists())
_all_checks_passed &= _check("Service file was written and syntax-checked", service_py_path.exists())
_all_checks_passed &= _check("Reproduction matches Notebook 67's reported values (cuts + correlation)",
                              REPRODUCTION_PASSED)
_all_checks_passed &= _check("Persisted profile verified against a fresh reproduction sample", PROFILE_VERIFIED)
_all_checks_passed &= _check("Bootstrap Spearman correlation CI is well-formed (lower <= point <= upper)",
                              SPEARMAN_CORR_CI[0] <= REPRODUCED_SPEARMAN_CORR <= SPEARMAN_CORR_CI[1])
_all_checks_passed &= _check("API self-test passed on all sampled real holdout customers", API_SELF_TEST_PASSED)
_all_checks_passed &= _check("Reproduced tier cuts are correctly ordered (low < high)",
                              REPRODUCED_CUT_LOW < REPRODUCED_CUT_HIGH)
_expected_files = [deployment_policy_path, service_py_path,
                   API_SUBDIR / ".env.example", API_SUBDIR / "requirements-api.txt"]
for _fp in _expected_files:
    _all_checks_passed &= _check(f"{_fp.name} exists and is non-empty", _fp.exists() and _fp.stat().st_size > 0)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 12 complete -- all checks passed.")


# =============================================================================
# SECTION 13: WRITE NOTEBOOK 68 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 13: Write Notebook 68 Summary Artifact")

NB68_SUMMARY = {
    "notebook": "68_profitability_modeling_validation_deployment.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "deployment_policy_path": str(deployment_policy_path),
    "service_py_path": str(service_py_path),
    "reproduction_passed": REPRODUCTION_PASSED,
    "profile_verified": PROFILE_VERIFIED,
    "reproduced_spearman_correlation": REPRODUCED_SPEARMAN_CORR,
    "spearman_correlation_ci_95": SPEARMAN_CORR_CI,
    "meets_kpi_with_ci": MEETS_KPI_WITH_CI,
    "recommended_for_production": PROFITABILITY_DEPLOYMENT_POLICY["recommended_for_production"],
    "api_self_test_passed": API_SELF_TEST_PASSED,
    "random_seed": RANDOM_SEED,
}
NB68_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_68_summary.json"
with open(NB68_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB68_SUMMARY, f, indent=2)
print(f"Wrote: {NB68_SUMMARY_PATH}")

_section("NOTEBOOK 68 COMPLETE")
print(f"Reproduction passed                     : {REPRODUCTION_PASSED}")
print(f"Profile verified                        : {PROFILE_VERIFIED}")
print(f"risk_adjustment_materiality CI           : [{SPEARMAN_CORR_CI[0]:.4f}, {SPEARMAN_CORR_CI[1]:.4f}]")
print(f"profitability_tier_monotonicity (repro)  : {PROFITABILITY_TIER_MONOTONICITY_REPRO_PASSED}")
print(f"RECOMMENDED_FOR_PRODUCTION (this run)    : {PROFITABILITY_DEPLOYMENT_POLICY['recommended_for_production']}")
print(f"API self-test passed                     : {API_SELF_TEST_PASSED}")
print(f"Deployment policy written to: {deployment_policy_path}")
print(f"Service written to          : {service_py_path}")
print(
    "\nNext: 69_profitability_modeling_financial_impact_reporting_packaging.ipynb -- synthesizes real results "
    "from all three prior notebooks (66-68), prices the real financial value of this profitability scoring "
    "system, packages the Word/Excel/HTML reports, and closes out Problem 13."
)